# Transformer Recommender на MovieLens 1M

Этот notebook собирает весь проект в один воспроизводимый pipeline:

1. загрузка и temporal preprocessing данных;
2. `Dataset` и `DataLoader` для последовательностей;
3. инициализация `TransformerRecommender`;
4. загрузка готового checkpoint или обучение, только если checkpoint отсутствует;
5. validation/test evaluation с `HR@10`, `NDCG@10` и full-history seen-item masking;
6. popularity baseline и честное сравнение;
7. пример рекомендаций с названиями фильмов.

Notebook рассчитан на **Restart Kernel → Run All**. Временные `reload`-ячейки не нужны: после перезапуска ядра модули из `src` импортируются заново. Существующий `checkpoints/best_model.pt` загружается автоматически, поэтому модель без необходимости не переобучается.


## 1. Imports и конфигурация

Поиск корня проекта делает notebook устойчивым к запуску как из корня репозитория, так и из папки `notebooks/`. Приоритет устройства: Apple MPS → CUDA → CPU.


In [ ]:
from pathlib import Path
import math
import random
import sys

import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader


def find_project_root(start: Path) -> Path:
    # Ищем ближайшую родительскую папку, содержащую src/.
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Не найден корень проекта с папкой src/. "
        "Запусти notebook внутри репозитория transformer-recommender."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset import MovieSequenceDataset
from src.data.load_data import load_movielens
from src.data.preprocess import (
    build_user_histories,
    generate_sequences,
    generate_test_sequences,
    generate_validation_sequences,
    sort_interactions,
    temporal_split,
)
from src.evaluation.metrics import evaluate_model
from src.models.transformer_recommender import TransformerRecommender


SEED = 42
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ml-1m"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
BEST_CHECKPOINT = CHECKPOINT_DIR / "best_model.pt"

MAX_SEQ_LEN = 50
BATCH_SIZE = 64
EMBEDDING_DIM = 128
NUM_HEADS = 4
HIDDEN_DIM = 512
NUM_LAYERS = 2
LEARNING_RATE = 1e-3
NUM_EPOCHS = 6
TOP_K = 10

# False = использовать готовый best_model.pt, если он существует.
FORCE_RETRAIN = False

random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", device)


## 2. Загрузка и preprocessing MovieLens

Для каждого пользователя взаимодействия сортируются по времени. Последние два фильма откладываются как validation- и test-target:

```text
ранняя история → train | предпоследний фильм → validation | последний фильм → test
```

Такой split имитирует реальную задачу: предсказать следующее взаимодействие, используя только прошлое. Исходный `movie_id` преобразуется в плотный `movie_idx` от `0` до `num_movies - 1`; отдельный индекс `num_movies` используется только для padding.


In [ ]:
ratings, movies = load_movielens(DATA_DIR)
ratings = sort_interactions(ratings.copy())

movie_ids = sorted(ratings["movie_id"].unique())
movie2idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
idx2movie = {idx: movie_id for movie_id, idx in movie2idx.items()}

ratings["movie_idx"] = ratings["movie_id"].map(movie2idx)

num_movies = len(movie2idx)
padding_idx = num_movies

user_histories = build_user_histories(ratings)
train_histories, val_targets, test_targets = temporal_split(user_histories)

print(f"Ratings: {len(ratings):,}")
print(f"Users: {len(user_histories):,}")
print(f"Movies: {num_movies:,}")
print(f"Padding index: {padding_idx}")


In [ ]:
train_sequences = generate_sequences(
    train_histories,
    max_seq_len=MAX_SEQ_LEN,
)
val_sequences = generate_validation_sequences(
    train_histories,
    val_targets,
)
test_sequences = generate_test_sequences(
    train_histories=train_histories,
    val_targets=val_targets,
    test_targets=test_targets,
)

print(f"Train sequences: {len(train_sequences):,}")
print(f"Validation examples: {len(val_sequences):,}")
print(f"Test examples: {len(test_sequences):,}")


## 3. Dataset и DataLoader

Актуальный `MovieSequenceDataset` возвращает три значения:

```python
x, y, user_id
```

- `x` — последние `MAX_SEQ_LEN` фильмов с padding слева;
- `y` — следующий фильм;
- `user_id` — нужен evaluation-коду, чтобы исключить **всю** известную историю пользователя, а не только последние 50 элементов из `x`.

В training loop `user_id` не используется, поэтому batch распаковывается как `x_batch, y_batch, _`.


In [ ]:
train_dataset = MovieSequenceDataset(
    sequences=train_sequences,
    max_seq_len=MAX_SEQ_LEN,
    padding_idx=padding_idx,
)
val_dataset = MovieSequenceDataset(
    sequences=val_sequences,
    max_seq_len=MAX_SEQ_LEN,
    padding_idx=padding_idx,
)
test_dataset = MovieSequenceDataset(
    sequences=test_sequences,
    max_seq_len=MAX_SEQ_LEN,
    padding_idx=padding_idx,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

x_batch, y_batch, user_ids = next(iter(train_loader))

print("X shape:", x_batch.shape)
print("y shape:", y_batch.shape)
print("user_ids shape:", user_ids.shape)
print("Train batches:", len(train_loader))


## 4. TransformerRecommender

Модель получает последовательность `movie_idx` и превращает каждый фильм в обучаемый embedding. Positional embedding добавляет информацию о порядке. Transformer смешивает контекст всей последовательности, а последний contextual vector проецируется в logits для всех фильмов:

```text
movie_idx → movie embedding + position embedding
          → Transformer encoder
          → контекст последней позиции
          → Linear(embedding_dim, num_movies)
          → logits
```

Жанры не подаются модели как признаки: связи между фильмами она учит из последовательностей поведения пользователей.


In [ ]:
model = TransformerRecommender(
    num_movies=num_movies,
    embedding_dim=EMBEDDING_DIM,
    max_seq_len=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

x_check, y_check, _ = next(iter(train_loader))
x_check = x_check.to(device)
y_check = y_check.to(device)

model.eval()
with torch.no_grad():
    logits_check = model(x_check)
    initial_loss = criterion(logits_check, y_check)

print("Logits shape:", logits_check.shape)
print(f"Loss before training/loading: {initial_loss.item():.4f}")
print("Model device:", next(model.parameters()).device)


## 5. Обучение и checkpointing

Если `checkpoints/best_model.pt` уже существует, веса загружаются и обучение пропускается. Чтобы намеренно начать обучение заново, нужно явно установить `FORCE_RETRAIN = True` в конфигурации.

Во время обучения лучшая эпоха выбирается **только по validation NDCG@10**. Test не участвует в подборе модели. В training loop третий элемент batch корректно игнорируется через `_`.


In [ ]:
def train_one_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for x_batch, y_batch, _ in data_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


def load_checkpoint(path, model, device):
    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    return checkpoint


In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if BEST_CHECKPOINT.exists() and not FORCE_RETRAIN:
    checkpoint = load_checkpoint(BEST_CHECKPOINT, model, device)
    print(f"Loaded checkpoint: {BEST_CHECKPOINT}")
    print("Loaded epoch:", checkpoint.get("epoch", "unknown"))
    print("Training skipped. Set FORCE_RETRAIN=True to retrain explicitly.")
else:
    best_ndcg = float("-inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
        )

        # На validation известна только train-history; val-target не маскируем.
        val_hr, val_ndcg = evaluate_model(
            model=model,
            data_loader=val_loader,
            device=device,
            seen_histories=train_histories,
            k=TOP_K,
        )

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "hit_rate@10": val_hr,
            "ndcg@10": val_ndcg,
        }

        torch.save(checkpoint, CHECKPOINT_DIR / f"epoch_{epoch}.pt")

        if val_ndcg > best_ndcg:
            best_ndcg = val_ndcg
            torch.save(checkpoint, BEST_CHECKPOINT)

        print(
            f"Epoch {epoch}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val HR@{TOP_K}: {val_hr:.4f} | "
            f"Val NDCG@{TOP_K}: {val_ndcg:.4f}"
        )

    checkpoint = load_checkpoint(BEST_CHECKPOINT, model, device)
    print(f"Loaded best epoch: {checkpoint['epoch']}")


## 6. Evaluation и full-history seen-item masking

`HR@10` отвечает на вопрос: попал ли настоящий следующий фильм хотя бы в Top-10. `NDCG@10` дополнительно награждает модель за более высокую позицию target в этом списке.

Модель получает контекст максимум из 50 последних фильмов, но фильтрация кандидатов использует **полную доступную историю**:

```text
validation: mask = train history
test:       mask = train history + validation target
```

Test-target нельзя добавлять в `test_seen_histories`: иначе правильный ответ будет замаскирован до оценки. `evaluate_model` получает `user_id` из `Dataset`, находит соответствующую историю и исключает её из logits перед `topk`.


In [ ]:
val_seen_histories = {
    user_id: list(train_history)
    for user_id, train_history in train_histories.items()
}

test_seen_histories = {
    user_id: train_histories[user_id] + [val_targets[user_id]]
    for user_id in train_histories
}

example_user_id = next(iter(test_seen_histories))
assert test_targets[example_user_id] not in test_seen_histories[example_user_id]

print(
    "Known items before test for example user:",
    len(test_seen_histories[example_user_id]),
)


In [ ]:
val_hr, val_ndcg = evaluate_model(
    model=model,
    data_loader=val_loader,
    device=device,
    seen_histories=val_seen_histories,
    k=TOP_K,
)

print(f"Validation HR@{TOP_K}:   {val_hr:.4f}")
print(f"Validation NDCG@{TOP_K}: {val_ndcg:.4f}")


## 7. Финальный test

Checkpoint уже выбран по validation. Test используется один раз для финальной оценки и не влияет на выбор эпохи.


In [ ]:
transformer_hr, transformer_ndcg = evaluate_model(
    model=model,
    data_loader=test_loader,
    device=device,
    seen_histories=test_seen_histories,
    k=TOP_K,
)

print(f"Transformer HR@{TOP_K}:   {transformer_hr:.4f}")
print(f"Transformer NDCG@{TOP_K}: {transformer_ndcg:.4f}")


Для текущего сохранённого checkpoint контрольные результаты с full-history masking:

| Модель | HR@10 | NDCG@10 |
|---|---:|---:|
| Transformer | **0.2954** | **0.1714** |

Если после Run All получаются другие числа, сначала проверь, что загружен тот же `best_model.pt` и что текущая версия `src.evaluation.metrics.evaluate_model` маскирует полную историю по `user_id`.


## 8. Popularity baseline

Baseline считает частоту фильмов **только по train-историям**, чтобы не подсматривать validation/test. Для каждого пользователя из глобального рейтинга удаляются уже просмотренные фильмы. Протокол test тем самым совпадает с Transformer:

```text
train history + validation target → исключить seen → Top-10 → test target
```


In [ ]:
movie_counts = {}

for history in train_histories.values():
    for movie_idx in history:
        movie_counts[movie_idx] = movie_counts.get(movie_idx, 0) + 1

popular_movies = sorted(
    movie_counts,
    key=movie_counts.get,
    reverse=True,
)


def recommend_popular(history, popular_movies, k=10):
    seen = set(history)
    recommendations = []

    for movie_idx in popular_movies:
        if movie_idx not in seen:
            recommendations.append(movie_idx)
        if len(recommendations) == k:
            break

    return recommendations


def evaluate_popularity(
    train_histories,
    val_targets,
    test_targets,
    popular_movies,
    k=10,
):
    total_hits = 0
    total_ndcg = 0.0

    for user_id, train_history in train_histories.items():
        known_history = train_history + [val_targets[user_id]]
        target = test_targets[user_id]
        recommendations = recommend_popular(
            history=known_history,
            popular_movies=popular_movies,
            k=k,
        )

        if target in recommendations:
            total_hits += 1
            rank = recommendations.index(target) + 1
            total_ndcg += 1 / math.log2(rank + 1)

    total_users = len(test_targets)
    return total_hits / total_users, total_ndcg / total_users


popularity_hr, popularity_ndcg = evaluate_popularity(
    train_histories=train_histories,
    val_targets=val_targets,
    test_targets=test_targets,
    popular_movies=popular_movies,
    k=TOP_K,
)

print(f"Popularity HR@{TOP_K}:   {popularity_hr:.4f}")
print(f"Popularity NDCG@{TOP_K}: {popularity_ndcg:.4f}")


## 9. Сравнение моделей

Обе модели оцениваются на одинаковом temporal test split и исключают всю историю, известную на момент предсказания.


In [ ]:
comparison = pd.DataFrame(
    {
        "Model": ["Transformer", "Popularity"],
        f"HR@{TOP_K}": [transformer_hr, popularity_hr],
        f"NDCG@{TOP_K}": [transformer_ndcg, popularity_ndcg],
    }
).set_index("Model")

comparison.style.format("{:.4f}")


Контрольные test-результаты проекта:

| Модель | HR@10 | NDCG@10 |
|---|---:|---:|
| Transformer + full-history masking | **0.2954** | **0.1714** |
| Popularity baseline | 0.0369 | 0.0180 |

Transformer примерно в 8 раз чаще помещает настоящий следующий фильм в Top-10, а более высокий NDCG показывает, что target обычно оказывается и ближе к вершине ranking.


## 10. Пример рекомендаций по названиям

После завершения test для демонстрации можно использовать всю известную историю пользователя: train + validation + test. Все просмотренные фильмы маскируются перед `topk`. Порядок рекомендаций модели сохраняется при переводе `movie_idx → movie_id → title`.


In [ ]:
def pad_history(history, max_seq_len, padding_idx):
    sequence = history[-max_seq_len:]
    return [padding_idx] * (max_seq_len - len(sequence)) + sequence


def recommend_with_transformer(
    model,
    history,
    device,
    max_seq_len,
    padding_idx,
    k=10,
):
    sequence = pad_history(history, max_seq_len, padding_idx)
    x = torch.tensor(sequence, dtype=torch.long, device=device).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        scores = model(x).squeeze(0)
        seen_movies = torch.tensor(
            sorted(set(history)),
            dtype=torch.long,
            device=device,
        )
        scores[seen_movies] = torch.finfo(scores.dtype).min
        return torch.topk(scores, k=k).indices.cpu().tolist()


In [ ]:
user_id = 1
full_history = (
    train_histories[user_id]
    + [val_targets[user_id]]
    + [test_targets[user_id]]
)

recommended_indices = recommend_with_transformer(
    model=model,
    history=full_history,
    device=device,
    max_seq_len=MAX_SEQ_LEN,
    padding_idx=padding_idx,
    k=TOP_K,
)

movie_info = movies.set_index("movie_id")

recent_history_rows = []
for movie_idx in full_history[-10:]:
    movie_id = idx2movie[movie_idx]
    recent_history_rows.append(
        {
            "title": movie_info.loc[movie_id, "title"],
            "genres": movie_info.loc[movie_id, "genres"],
        }
    )

recommendation_rows = []
for rank, movie_idx in enumerate(recommended_indices, start=1):
    movie_id = idx2movie[movie_idx]
    recommendation_rows.append(
        {
            "rank": rank,
            "title": movie_info.loc[movie_id, "title"],
            "genres": movie_info.loc[movie_id, "genres"],
        }
    )

print(f"Последние фильмы пользователя {user_id}:")
display(pd.DataFrame(recent_history_rows))

print(f"Top-{TOP_K} рекомендаций:")
display(pd.DataFrame(recommendation_rows).set_index("rank"))


## Вывод

Pipeline отделяет три разные задачи:

- Transformer использует последние 50 фильмов как контекст;
- evaluation исключает всю известную историю из кандидатов;
- test остаётся независимым от обучения и выбора checkpoint.

Именно одинаковая фильтрация кандидатов делает сравнение Transformer и popularity baseline корректным.
